In [ ]:
%%capture
!pip install unsloth

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA = "/content/drive/MyDrive/selfrag"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=f"{DATA}/lora_adapter_v2",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

==((====))==  Unsloth 2026.11.9: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth: Already have LoRA adapters! We shall skip this step.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files={
    "train": f"{DATA}/sft_train.jsonl",
    "validation": f"{DATA}/sft_val.jsonl",
})


def to_text(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}


ds = ds.map(to_text)
print(ds)
print(ds["train"][0]["text"][:600])

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1975 [00:00<?, ? examples/s]

Map:   0%|          | 0/233 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages', 'text'],
        num_rows: 1975
    })
    validation: Dataset({
        features: ['messages', 'text'],
        num_rows: 233
    })
})
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
TASK: RETRIEVE
Decide whether this question needs the Indian Railways rule manuals to be answered. Reply Yes or No.

Question: How should goods to and from the out-agency be recorded on luggage tickets and invoices?<|im_end|>
<|im_start|>assistant
Yes<|im_end|>



In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq


def tokenize(example):
    user_msg = example["messages"][0]
    answer = example["messages"][1]["content"]

    prompt = tokenizer.apply_chat_template(
        [user_msg], tokenize=False, add_generation_prompt=True
    )
    full = prompt + answer + "<|im_end|>"

    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full, add_special_tokens=False)["input_ids"]

    # -100 means "ignore": the model is graded only on the answer part
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }


train_tok = ds["train"].map(tokenize, remove_columns=ds["train"].column_names)
print("Examples to train on:", len(train_tok))
print("Examples with no answer tokens:",
      sum(all(l == -100 for l in ex["labels"]) for ex in train_tok))

trainer = Trainer(
    model=model,
    train_dataset=train_tok,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tokenizer, padding=True, label_pad_token_id=-100
    ),
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=1,
                    # QUICK TEST. Delete this line for the real run.
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=f"{DATA}/outputs_v2",
        save_steps=100,
        save_total_limit=1,
        report_to="none",
    ),
)

Examples to train on: 1975
Examples with no answer tokens: 0


In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,975 | Num Epochs = 1 | Total steps = 247
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Step,Training Loss
5,0.434139
10,0.459013
15,0.510283
20,0.504919
25,0.605614
30,0.475375
35,0.520090
40,0.382656
45,0.479304
50,0.394833


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/selfrag/outputs_v2/checkpoint-100/tokenizer_config.json.


Step,Training Loss
5,0.434139
10,0.459013
15,0.510283
20,0.504919
25,0.605614
30,0.475375
35,0.520090
40,0.382656
45,0.479304
50,0.394833


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/selfrag/outputs_v2/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/selfrag/outputs_v2/checkpoint-247/tokenizer_config.json.


TrainOutput(global_step=247, training_loss=0.394928658539467, metrics={'train_runtime': 1563.1978, 'train_samples_per_second': 1.263, 'train_steps_per_second': 0.158, 'total_flos': 2.7194492737019904e+16, 'train_loss': 0.394928658539467, 'epoch': 1.0})

In [ ]:
model.save_pretrained(f"{DATA}/lora_adapter_v2")
tokenizer.save_pretrained(f"{DATA}/lora_adapter_v2")
print("Saved")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/selfrag/lora_adapter_v2/tokenizer_config.json.


Saved


In [ ]:
from collections import Counter

print(ds)
print(Counter(ex["messages"][0]["content"].split("\n")[0] for ex in ds["train"]))
print("Lines in the file on Drive:",
      sum(1 for _ in open(f"{DATA}/sft_train.jsonl", encoding="utf-8")))

DatasetDict({
    train: Dataset({
        features: ['messages', 'text'],
        num_rows: 1975
    })
    validation: Dataset({
        features: ['messages', 'text'],
        num_rows: 233
    })
})
Counter({'TASK: RELEVANCE': 1081, 'TASK: ANSWER': 358, 'TASK: SUPPORT': 353, 'TASK: RETRIEVE': 183})
Lines in the file on Drive: 1975


In [ ]:
import json
import collections

FastLanguageModel.for_inference(model)

val = [json.loads(l) for l in open(f"{DATA}/sft_val.jsonl", encoding="utf-8")]
scores = collections.defaultdict(lambda: [0, 0])

for ex in val:
    prompt = ex["messages"][0]["content"]
    target = ex["messages"][1]["content"]
    task = prompt.split("\n")[0]
    if task == "TASK: ANSWER":
        continue

    text = tokenizer.apply_chat_template(
        [ex["messages"][0]], tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=5, do_sample=False)
    pred = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

    scores[task][1] += 1
    if pred.lower().split()[:1] == [target.lower()]:
        scores[task][0] += 1

for task, (ok, total) in scores.items():
    print(f"{task}: {ok}/{total} correct ({100 * ok / total:.0f}%)")

Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TASK: RELEVANCE: 90/119 correct (76%)
TASK: RETRIEVE: 27/27 correct (100%)
TASK: SUPPORT: 39/45 correct (87%)


In [ ]:
import collections

lens = collections.defaultdict(list)
for ex in ds["train"]:
    task = ex["messages"][0]["content"].split("\n")[0]
    lens[task].append(len(tokenizer(ex["text"])["input_ids"]))

for task, L in lens.items():
    over = sum(n > 2048 for n in L)
    print(f"{task}: {len(L)} examples, longest {max(L)} tokens, over 2048: {over}")

try:
    kept = collections.Counter()
    for ex in trainer.train_dataset:
        text = tokenizer.decode(ex["input_ids"])
        for t in ["RETRIEVE", "RELEVANCE", "ANSWER", "SUPPORT"]:
            if "TASK: " + t in text:
                kept[t] += 1
    print("Kept by the trainer:", len(trainer.train_dataset), dict(kept))
except Exception as e:
    print("Could not count kept examples:", e)

TASK: RETRIEVE: 183 examples, longest 96 tokens, over 2048: 0
TASK: ANSWER: 358 examples, longest 1692 tokens, over 2048: 0
TASK: RELEVANCE: 1081 examples, longest 858 tokens, over 2048: 0
TASK: SUPPORT: 353 examples, longest 1857 tokens, over 2048: 0
Kept by the trainer: 1975 {'RETRIEVE': 183, 'ANSWER': 358, 'RELEVANCE': 1081, 'SUPPORT': 353}


In [ ]:
import json
import os

from unsloth import FastLanguageModel
from transformers.utils import logging
logging.set_verbosity_error()

FastLanguageModel.for_inference(model)

NOT_FOUND = "I cannot find this in the documents."


def numbered(passages):
    return "\n\n".join(f"[{i}] {t}" for i, t in enumerate(passages, start=1))


def retrieve_prompt(question):
    return (
        "TASK: RETRIEVE\n"
        "Decide whether this question needs the Indian Railways rule manuals to be answered. "
        "Reply Yes or No.\n\n"
        f"Question: {question}"
    )


def relevance_prompt(question, passage):
    return (
        "TASK: RELEVANCE\n"
        "Decide whether the passage contains information that helps answer this exact question. "
        "Reply Relevant or Irrelevant.\n\n"
        f"Question: {question}\n\nPassage: {passage}"
    )


def generate_prompt(question, passages):
    return (
        "TASK: ANSWER\n"
        "Answer the question using the passages. Keep it to 1-3 sentences.\n\n"
        f"Passages:\n{numbered(passages)}\n\nQuestion: {question}"
    )


def support_prompt(question, answer, passages):
    return (
        "TASK: SUPPORT\n"
        "Decide whether every claim in the answer, including numbers, is stated in the passages, "
        "and the answer directly answers the question that was asked. "
        "Reply Supported or Unsupported.\n\n"
        f"Question: {question}\n\nAnswer: {answer}\n\nPassages:\n{numbered(passages)}"
    )


def run(user_text, max_new_tokens):
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_text}],
        tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()


rows = json.load(open(f"{DATA}/eval_retrieved.json", encoding="utf-8"))
OUT = f"{DATA}/results_finetuned_selfrag.jsonl"

done = set()
if os.path.exists(OUT):
    with open(OUT, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                done.add(json.loads(line)["id"])

with open(OUT, "a", encoding="utf-8") as out_file:
    for n, r in enumerate(rows, start=1):
        if r["id"] in done:
            continue

        q = r["question"]
        passages = r["passages"]

        needs = run(retrieve_prompt(q), 3).lower().startswith("yes")
        relevant, draft, support = [], None, "none"

        if not needs:
            final = run(q, 150)
        else:
            relevant = [
                p for p in passages
                if run(relevance_prompt(q, p["text"]), 4).lower().startswith("relevant")
            ]
            if not relevant:
                final = NOT_FOUND
            else:
                texts = [p["text"] for p in relevant[:3]]
                draft = run(generate_prompt(q, texts), 200)
                verdict = run(support_prompt(q, draft, texts), 4).lower()
                supported = verdict.startswith("supported")
                support = "fully" if supported else "no"
                final = draft if supported else NOT_FOUND

        out_file.write(json.dumps({
            "id": r["id"],
            "question": q,
            "label": r["label"],
            "gold_answer": r["gold_answer"],
            "gold_chunk": r["gold_chunk"],
            "retrieved": [p["id"] for p in passages],
            "relevant": [p["id"] for p in relevant],
            "draft": draft,
            "support": support,
            "model_answer": final,
        }, ensure_ascii=False) + "\n")
        out_file.flush()
        print(n, "done")

print("All done. Results in", OUT)

1 done
2 done
3 done
4 done
5 done
6 done
7 done
8 done
9 done
10 done
11 done
12 done
13 done
14 done
15 done
16 done
17 done
18 done
19 done
20 done
21 done
22 done
23 done
24 done
25 done
26 done
27 done
28 done
29 done
30 done
31 done
32 done
33 done
34 done
35 done
36 done
37 done
38 done
39 done
40 done
41 done
42 done
43 done
44 done
45 done
46 done
47 done
48 done
49 done
50 done
51 done
52 done
53 done
54 done
55 done
56 done
57 done
58 done
59 done
60 done
61 done
62 done
63 done
64 done
65 done
66 done
67 done
68 done
69 done
70 done
71 done
72 done
73 done
74 done
75 done
76 done
77 done
78 done
79 done
80 done
81 done
82 done
83 done
84 done
85 done
86 done
87 done
88 done
89 done
90 done
91 done
92 done
93 done
94 done
95 done
96 done
97 done
98 done
99 done
100 done
101 done
102 done
103 done
104 done
105 done
106 done
107 done
108 done
109 done
110 done
111 done
112 done
113 done
114 done
115 done


In [ ]:
import json

OUT = f"{DATA}/results_finetuned_selfrag.jsonl"
rows = json.load(open(f"{DATA}/eval_retrieved.json", encoding="utf-8"))
print("Questions to process:", len(rows))
print("Results saved so far:", sum(1 for _ in open(OUT, encoding="utf-8")))

Questions to process: 159
Results saved so far: 159


In [ ]:
import json
import os

from unsloth import FastLanguageModel
from transformers.utils import logging
logging.set_verbosity_error()

FastLanguageModel.for_inference(model)


def numbered(passages):
    return "\n\n".join(f"[{i}] {t}" for i, t in enumerate(passages, start=1))


def support_prompt(question, answer, passages):
    return (
        "TASK: SUPPORT\n"
        "Decide whether every claim in the answer, including numbers, is stated in the passages, "
        "and the answer directly answers the question that was asked. "
        "Reply Supported or Unsupported.\n\n"
        f"Question: {question}\n\nAnswer: {answer}\n\nPassages:\n{numbered(passages)}"
    )


def run(user_text, max_new_tokens):
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_text}],
        tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()


drafts = json.load(open(f"{DATA}/hybrid_drafts.json", encoding="utf-8"))
OUT = f"{DATA}/hybrid_support.jsonl"

done = set()
if os.path.exists(OUT):
    with open(OUT, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                done.add(json.loads(line)["id"])

with open(OUT, "a", encoding="utf-8") as out_file:
    for n, d in enumerate(drafts, start=1):
        if d["id"] in done:
            continue
        if d["draft"] is None or not d["texts"]:
            label = "none"
        else:
            verdict = run(support_prompt(d["question"], d["draft"], d["texts"]), 4).lower()
            label = "fully" if verdict.startswith("supported") else "no"
        out_file.write(json.dumps({"id": d["id"], "support": label}) + "\n")
        out_file.flush()

print("Finished", len(drafts), "items. Results in", OUT)

Finished 159 items. Results in /content/drive/MyDrive/selfrag/hybrid_support.jsonl
